# AF2 + CPE0 seed-42 matched screen
Runs static audit, AF2CPE0 loss-weight-zero control, AF2CPE5 candidate, then frozen validation decision. Locked test is never exposed. Run with a GPU runtime.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os, shutil, subprocess, sys, tarfile
from pathlib import Path
REPO=Path('/content/coffee-bean-detection'); BRANCH='codex/af2-cpe0-seed42'
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO)],check=True)
os.chdir(REPO); sys.path.insert(0,str(REPO/'src'))
from coffee_detector.drive_project import resolve_drive_project_root, require_project_artifact
REQ=('bundles/faruq-development-v3-grouped.tar','experiments/faruq-v3-breadth-screening-batch-v1/candidates/AFAB/AF2_seed42/weights/best.pt')
PROJECT=resolve_drive_project_root(required_relative_paths=REQ); ARCHIVE=require_project_artifact(PROJECT,REQ[0]); AF2=require_project_artifact(PROJECT,REQ[1])
DATA=Path('/content/faruq-development-v3-grouped')
if not (DATA/'data.yaml').is_file():
    with tarfile.open(ARCHIVE,'r') as archive: archive.extractall('/content',filter='data')
assert not (DATA/'test').exists(); SUMMARY=DATA/'faruq_grouped_summary.json'; OUTPUT=PROJECT/'experiments/faruq-v3-af2-cpe0-seed42-v1'; AUDIT=OUTPUT/'static_audit.json'


In [ ]:
subprocess.run([sys.executable,'-m','coffee_detector.experiments.run_faruq_v3_af2_cpe_static','--af2-checkpoint',str(AF2),'--output',str(AUDIT),'--device','0'],check=True)
for arm in ('AF2CPE0','AF2CPE5'):
    subprocess.run([sys.executable,'-m','coffee_detector.experiments.run_faruq_v3_af2_cpe_arm','--arm',arm,'--data-root',str(DATA),'--grouped-summary',str(SUMMARY),'--af2-checkpoint',str(AF2),'--static-audit',str(AUDIT),'--output-root',str(OUTPUT),'--seed','42','--device','0','--authorize-training'],check=True)


In [ ]:
CONTROL=OUTPUT/'val_reports/AF2CPE0_seed42_result.json'; CANDIDATE=OUTPUT/'val_reports/AF2CPE5_seed42_result.json'; DECISION=OUTPUT/'decision_seed42.json'
subprocess.run([sys.executable,'-m','coffee_detector.experiments.run_faruq_v3_af2_cpe_decision','--control',str(CONTROL),'--candidate',str(CANDIDATE),'--output',str(DECISION)],check=True)
print(DECISION.read_text())
